In [ ]:
import requests
import pandas as pd

In [ ]:
base = "https://earthquake.usgs.gov/fdsnws/event/1/query"
params = {"format": "geojson", "starttime": "2024-01-01", "endtime": "2024-01-03", "minmagnitude": 5}

Concretely: minmagnitude=5 is a parameter. It never appears as a column. properties.mag is a column. You can't put it in a URL.


In [ ]:
r = requests.get(base, params=params)
print(r.status_code)

200


**Where did the parameters come from?**

From the provider's documentation. There is no way to derive them, guess them, or discover them from the response. The server decides what it accepts, and publishes that list. For USGS it's the parameter table at https://earthquake.usgs.gov/fdsnws/event/1/.

Two backup routes when docs are thin:

Send a wrong one and read the error. Most well-built APIs return 400 with a message naming the offending parameter. Try minmagnitud=5 (typo) and look at r.text.
Machine-readable spec. USGS publishes https://earthquake.usgs.gov/fdsnws/event/1/application.wadl, which lists every accepted parameter. Many APIs publish an OpenAPI/Swagger file that does the same.


In [ ]:
print(r.url)

https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2024-01-01&endtime=2024-01-03&minmagnitude=5


## Are parameters columns? No.

They are two different lists, learned from two different places, at two different times.

| Aspect | Parameters | Columns |
|---|---|---|
| What they are | your question | the shape of the answer |
| Where they live | in the URL, after `?` | in the JSON body |
| Who defines them | the provider, in docs | whatever the server sent back |
| How you learn them | read docs, or trigger a 400 | `print(keys)` / `json_normalize` |
| When | **before** the request | **after** the request |

They occasionally rhyme — `minmagnitude` is a parameter that filters on the field
that comes back as the column `properties.mag` — but that is a coincidence of
naming, not a rule. `format`, `orderby`, `limit` and `offset` are parameters that
correspond to no column at all.

In [ ]:
data = r.json()
print(list(data.keys()))

['type', 'metadata', 'features', 'bbox']


## Where did `features` come from?

**How I knew it:** GeoJSON is a published standard, not a USGS invention. Any
GeoJSON document is a `FeatureCollection` containing a list called `features`,
and every entry in it has `properties` and `geometry`. This is prior knowledge,
not something derived from the response.

**How to know it without prior knowledge — the probe:**

```python
print({k: type(v).__name__ for k, v in data.items()})
```

Output looks like `{'type': 'str', 'metadata': 'dict', 'features': 'list', 'bbox': 'list'}`.

The record list is the key whose value is a `list` **of dicts**. `bbox` is also a
list, but of four numbers — `len()` and inspecting element zero separates them
immediately.

This procedure works on any API, including one never seen before.

In [ ]:
print(data["metadata"])

{'generated': 1786549439000, 'url': 'https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2024-01-01&endtime=2024-01-03&minmagnitude=5', 'title': 'USGS Earthquakes', 'status': 200, 'api': '2.7.0', 'count': 16}


In [ ]:
rows = data["features"]
print(len(rows))

16


In [ ]:
rows[0]

{'type': 'Feature',
 'properties': {'mag': 5,
  'place': '97 km W of El Aguilar, Argentina',
  'time': 1704236220970,
  'updated': 1710020331040,
  'tz': None,
  'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/us6000m1ae',
  'detail': 'https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=us6000m1ae&format=geojson',
  'felt': None,
  'cdi': None,
  'mmi': None,
  'alert': None,
  'status': 'reviewed',
  'tsunami': 0,
  'sig': 385,
  'net': 'us',
  'code': '6000m1ae',
  'ids': ',us6000m1ae,',
  'sources': ',us,',
  'types': ',moment-tensor,origin,phase-data,',
  'nst': 142,
  'dmin': 1.495,
  'rms': 0.94,
  'gap': 43,
  'magType': 'mww',
  'type': 'earthquake',
  'title': 'M 5.0 - 97 km W of El Aguilar, Argentina'},
 'geometry': {'type': 'Point', 'coordinates': [-66.6178, -23.3669, 188.635]},
 'id': 'us6000m1ae'}

## How the column names were found

They came from running `pd.json_normalize(rows)` and printing `.columns` —
not from documentation and not from memory. The list is the truth for this
specific query on this specific run.

The dotted names are literal paths into the nested JSON. `properties.mag` means:

```python
print(data["features"][0]["properties"]["mag"])
```

`json_normalize` did exactly that for every nested key. Verify the correspondence:

```python
print(rows[0]["properties"]["place"])
print(df["properties.place"].iloc[0])
```

Two remaining problems in the normalized frame:

- `geometry.coordinates` is **not flat** — it holds a list of
  `[longitude, latitude, depth]`. `json_normalize` stops at lists and does not
  expand them into columns.
- Several columns are mostly empty for most queries (`felt`, `cdi`, `mmi`,
  `alert`, `tz`). Check before selecting:

```python
print(df.isna().mean().sort_values(ascending=False).head(10))
```

In [ ]:
print(pd.DataFrame(rows).columns.tolist())

['type', 'properties', 'geometry', 'id']


In [ ]:
df = pd.json_normalize(rows)
print(df.columns.tolist())

['type', 'id', 'properties.mag', 'properties.place', 'properties.time', 'properties.updated', 'properties.tz', 'properties.url', 'properties.detail', 'properties.felt', 'properties.cdi', 'properties.mmi', 'properties.alert', 'properties.status', 'properties.tsunami', 'properties.sig', 'properties.net', 'properties.code', 'properties.ids', 'properties.sources', 'properties.types', 'properties.nst', 'properties.dmin', 'properties.rms', 'properties.gap', 'properties.magType', 'properties.type', 'properties.title', 'geometry.type', 'geometry.coordinates']


In [ ]:
import time

In [ ]:
params = {"format": "geojson", "starttime": "2020-01-01", "endtime": "2020-12-31", "minmagnitude": 4.5, "orderby": "time-asc", "limit": 1000}
frames = []

In [ ]:
offset = 1
while True:
    params["offset"] = offset
    rows = requests.get(base, params=params).json()["features"]
    if not rows: break
    frames.append(pd.json_normalize(rows))
    offset += params["limit"]
    time.sleep(1)

In [ ]:
big = pd.concat(frames, ignore_index=True)
print(big.shape)

(6496, 30)


## What `pd.concat` does, and why it is used here

It stacks DataFrames vertically into one table.

```python
a = pd.DataFrame({"x": [1, 2]})
b = pd.DataFrame({"x": [3, 4]})
```

```python
print(pd.concat([a, b], ignore_index=True))
```

Result: four rows. `ignore_index=True` renumbers the index 0,1,2,3. Without it
the index reads 0,1,0,1 — duplicate index values cause problems later.

Concatenating once at the end is also much faster than appending inside the loop.

In [ ]:
big["time"] = pd.to_datetime(big["properties.time"], unit="ms")
print(big["time"].min(), big["time"].max())

2020-01-01 00:28:20.289000 2020-12-30 23:03:00.251000


## Where the API actually is

It is a program running on the organisation's servers. They chose to expose it
and to publish the rules for using it.

The URL has three parts:

```
https://earthquake.usgs.gov  /fdsnws/event/1/query  ?format=geojson&starttime=...
        whose computer            which program           your question
```

To find out whether an organisation offers one, look on their site for "API",
"developer", "web services", or "data services". Government science agencies
almost always publish one; commercial platforms usually require a key.

`fdsnws` stands for FDSN Web Services — an **international standard** for seismic
data. Other agencies (IRIS, EMSC, GEOFON) expose the same endpoint shape with the
same parameters, so the same code works against them by changing only the
hostname. This kind of standardisation is common in scientific data and rare in
commercial APIs.

In [ ]:
big.to_csv("earthquakes.csv", index=False)